In [0]:

spark.conf.set("fs.azure.account.auth.type.<storage-account>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<storage-account>.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage-account>.dfs.core.windows.net", "<application-id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage-account>.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage-account>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory-id>/oauth2/token")

In [0]:
df_sales_17 = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferSchema", "true")\
          .load("abfs://bronze@adventureworksdatalakeom.dfs.core.windows.net/AdventureWorks_Sales_2017")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_sales_17.display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2017-01-01,2003-12-13,SO61285,529,23791,1,2,2
2017-01-01,2003-09-24,SO61285,214,23791,1,3,1
2017-01-01,2003-09-04,SO61285,540,23791,1,1,1
2017-01-01,2003-09-28,SO61301,529,16747,1,2,2
2017-01-01,2003-10-21,SO61301,377,16747,1,1,1
2017-01-01,2003-10-23,SO61301,540,16747,1,3,1
2017-01-01,2003-09-04,SO61269,215,11792,4,1,1
2017-01-01,2003-10-21,SO61269,229,11792,4,2,1
2017-01-01,2003-10-24,SO61286,528,11530,6,2,2
2017-01-01,2003-09-27,SO61286,536,11530,6,1,2


In [0]:
df_sales_17 = df_sales_17.withColumn(
    "OrderNumber",
    regexp_replace(col("OrderNumber"), "S", "T")
)

In [0]:
df_sales_17 = df_sales_17.groupBy("ProductKey").agg(
    sum("OrderQuantity").alias("TotalSold")
)

In [0]:
df_sales_17.display()

ProductKey,TotalSold
471,91
540,406
580,154
481,286
588,78
472,111
362,227
597,27
375,64
593,21


In [0]:
df_sales_17.write.format("parquet")\
            .mode("append")\
            .option("path","abfs://silver@adventureworksdatalakeom.dfs.core.windows.net/AdventureWorks_Sales_2017")\
            .save()